# Verifiable Quantum Circuit Compilation

## v8 · Invsqrt and DQI: Automatic Quadratic-Form Merges

Ehan Mallick removed two Toffolis from the best invsqrt circuit by hand: a stretch of the circuit where a qubit only
*collects* AND products by XOR is a quadratic form over GF(2), and Dickson's theorem says it needs only rank/2 ANDs.
One of the two merges also shares a product between two accumulators by CX fan-out.

**This notebook turns that idea into a scanner and runs it over the whole invsqrt and DQI circuits.**

- For every qubit, find each stretch where it only receives `CCX`/`CX`/`X` targets and nothing reads it.
- Write the collected polynomial symbolically (affine forms of the other wires) and rewrite it with `rank/2` ANDs.
- For pairs of accumulators, also try `f1 = f2 + (f1 + f2)` with the `f2` products fanned into both.
- A product may move in time past any op that leaves its target alone, except a measurement or reset (`HMR`, `R`)
  of a wire its value depends on; this is checked with a span test at every such op. Nothing under `PUSH_CONDITION`
  is touched. New CX/X only; no new qubits.
- Every candidate is checked by the **unmodified official verifiers**; invsqrt over its whole input domain.

The base is the portfolio from our notebook 07 (`T = 15,769` chemistry, expected GM ≈ 24.1806). The notebook always
writes a `submission.zip`: improved members where the verifier passed them, otherwise the base members unchanged.

| Task | Base cost | Value of one Toffoli |
|---|---:|---|
| invsqrt | Q·C = 62 × 1,622.07 | ≈ +0.005 LB |
| dqi | R·Q²·T, T ≈ 398k | ≈ +0.00002 LB |

**Kaggle setup:** Accelerator *None* (CPU only), Internet off. Inputs: the competition data and the output of our
notebook 07 (`flexonafft/dfthc-sos-angle-loader-search`). Runtime ≈ 30–60 minutes.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
SALT = 'flexonafft-v8-quadratic-merge'
BASE_ZIP_SHA = '73ae641e618af509129dbb36ec7dcb1232acfbad6c6c8544c05ce9b89fdfb87b'   # notebook 07 output
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439463583672.93, 'dfthcsos': 3355599588.3155375}
SCAN = {'invsqrt': dict(rounds=10, maxrun=10, maxspan=6000, pair_window=400, minutes=40),
        'dqi':     dict(rounds=6,  maxrun=6,  maxspan=3000, pair_window=150, minutes=90)}
WORKERS = 4

In [ ]:
# ── Session ───────────────────────────────────────────────────────────────────────────────────
import os, sys, glob, json, time, math, shutil, hashlib, zipfile
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')

def deep_find(pattern):
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / pattern), recursive=True))

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

KICK = deep_find('kickmix*.so')[0].parent
VERIF = deep_find('verify_submission_invsqrt.py')[0].parent.parent
sys.path[:0] = [str(KICK), str(VERIF)]
import kickmix as km
from invsqrt import verify_submission_invsqrt as vi
from dqi import verify_submission_dqi as vd
from dfthcsos import verify as vc
VERIFIER_FILES = sorted(p for p in VERIF.rglob('*.py'))
VERIFIER_SHA = {str(p): sha256(p) for p in VERIFIER_FILES}

zips = [p for p in deep_find('submission.zip') if sha256(p) == BASE_ZIP_SHA]
assert zips, f'notebook 07 submission.zip ({BASE_ZIP_SHA[:12]}…) is not attached: {deep_find("submission.zip")}'
BASE_ZIP = zips[0]
BASE_REPORT = json.loads((BASE_ZIP.parent / 'loader_search_A.json').read_text())
BASE = ROOT / 'base'; shutil.rmtree(BASE, ignore_errors=True)
with zipfile.ZipFile(BASE_ZIP) as z:
    z.extractall(BASE)
MEMBERS = sorted(p.relative_to(BASE).as_posix() for p in BASE.rglob('*') if p.is_file())
BASE_RESULTS = BASE_REPORT['verified_results']
print('base', BASE_ZIP, len(MEMBERS), 'members')
for t in ('invsqrt', 'dqi', 'dfthcsos'):
    r = BASE_RESULTS[t]
    print(f"  {t:8s} {r['status']}  cost {float(r['score']):.6g}  ratio {REFERENCE[t] / float(r['score']):.5f}  "
          f"Q {r['num_qubits']}  T {r['toffoli_count']}")

## The scanner

`qmerge.py` is written below and imported. The self-test re-runs Dickson's identity on random polynomials and
checks random wasteful X/CX/CCX/SWAP circuits on **every** input before and after rewriting. It also rebuilds Ehan
Mallick's second window (3 CCX, two accumulators, one shared product) from the bare pattern and finds the 3 → 2 merge.

In [ ]:
QMERGE_SOURCE = r'''"""qmerge: find XOR-accumulated AND products that can be re-synthesised with fewer Toffolis.

A qubit T that, over a stretch of the circuit, only receives CCX/CX/X targets and is not read, collects a GF(2)
polynomial of degree <= 2 in the values of the other wires. Dickson's theorem: such a quadratic form needs exactly
rank/2 AND gates. Two such targets can also share products: f1 = f2 + (f1 + f2), with the f2 products fanned out into
both targets by CX conjugation. This generalises Ehan Mallick's two hand-found 3->2 merges on invsqrt (qf2).

Semantics used (kickmix ops): X, CX, SWAP, CCX are classical permutations; Z, CZ, CCZ, Z_POW are diagonal (do not
change basis values); HMR and R discard a qubit; everything under PUSH_CONDITION, and every other op that touches a
qubit, is treated as an unknown change of that qubit. A product term may be moved in time past any op that does not
touch its target, except an HMR/R on a qubit its value depends on; this is checked by a span test at every such op.
"""
import re
from collections import defaultdict

QRE = re.compile(r'\bq(\d+)\b')
DIAG = {'Z', 'CZ', 'CCZ', 'Z_POW'}
PASS = {'INFO_TAG', 'DEBUG_PRINT', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1'}
DISCARD = {'HMR', 'R'}


# ── GF(2) polynomials of degree <= 2: (quad: set of (i<j), lin: set, const: int); affine form: (frozenset, int) ──
def mul(A, B):
    (a, a0), (b, b0) = A, B
    quad, lin = set(), set()
    for x in a:
        for y in b:
            if x == y:
                lin ^= {x}
            else:
                quad ^= {(x, y) if x < y else (y, x)}
    if a0:
        lin ^= set(b)
    if b0:
        lin ^= set(a)
    return quad, lin, a0 & b0


def add_into(P, Q):
    P[0].symmetric_difference_update(Q[0]); P[1].symmetric_difference_update(Q[1]); P[2] ^= Q[2]


def poly_of(entries):
    P = [set(), set(), 0]
    for e in entries:
        add_into(P, mul(e[1], e[2]))
    return P


def dickson(P):
    """f = sum_i L_i M_i + affine, with rank/2 products (minimal for one quadratic form)."""
    quad, lin, c = set(P[0]), set(P[1]), P[2]
    prods = []
    while quad:
        i, j = min(quad)
        A, B = set(), set()
        for (u, v) in [t for t in quad if i in t or j in t]:
            quad.discard((u, v))
            if (u, v) == (i, j):
                continue
            if i in (u, v):
                A ^= {v if u == i else u}
            else:
                B ^= {v if u == j else u}
        a0 = 1 if i in lin else 0
        b0 = 1 if j in lin else 0
        lin.discard(i); lin.discard(j)
        prods.append(((frozenset({i}) ^ frozenset(B), b0), (frozenset({j}) ^ frozenset(A), a0)))
        q2, l2, c2 = mul((frozenset(A), a0), (frozenset(B), b0))
        quad ^= q2; lin ^= l2; c ^= c2
    return prods, (frozenset(lin), c)


def evaluate(P, x):
    quad, lin, c = P
    s = c
    for (u, v) in quad:
        s ^= x[u] & x[v]
    for u in lin:
        s ^= x[u]
    return s


# ── Parsing ──
def parse(lines):
    ops = []
    for i, raw in enumerate(lines):
        s = raw.split('#', 1)[0].strip()
        if not s:
            continue
        tok = s.split(None, 1)
        op = tok[0].split('(', 1)[0]
        qs = [int(m) for m in QRE.findall(tok[1])] if len(tok) > 1 else []
        ops.append((i, op, qs))
    return ops


def stats(ops):
    h = defaultdict(int)
    for _, op, _ in ops:
        h[op] += 1
    return dict(h)


# ── Symbolic pass ──
class Tracker:
    def __init__(self):
        self.form = {}
        self.birth = []
        self.acc = {}

    def fresh(self, q, line):
        v = len(self.birth)
        self.birth.append(line)
        self.form[q] = (frozenset((v,)), 0)

    def get(self, q):
        if q not in self.form:
            self.fresh(q, -1)
        return self.form[q]


def run(ops, want=()):
    """One pass. Returns intervals (T, entries, start, end), discard events and snapshots at the lines in `want`."""
    want = set(want)
    t = Tracker()
    form, acc = t.form, t.acc
    intervals, discards, snaps = [], [], {}
    depth = 0
    for q in sorted({q for _, _, qs in ops for q in qs}):
        t.fresh(q, -1)

    def close(q, line):
        a = acc.pop(q, None)
        if a is not None:
            if a['entries']:
                intervals.append((q, a['entries'], a['start'], line))
            t.fresh(q, line)

    for (i, op, qs) in ops:
        if i in want:
            snaps[i] = ({q: f for q, f in form.items() if q not in acc}, frozenset(acc))
        if op == 'PUSH_CONDITION':
            depth += 1
            for q in qs:
                close(q, i)
            continue
        if op == 'POP_CONDITION':
            depth -= 1
            continue
        if op in PASS or not qs:
            continue
        for q in qs:
            t.get(q)
        if depth > 0:
            for q in qs:
                close(q, i)
                if op not in DIAG:
                    t.fresh(q, i)
            continue
        if op == 'X':
            for q in qs:
                if q not in acc:
                    fs, c = form[q]; form[q] = (fs, c ^ 1)
        elif op == 'CX' and len(qs) % 2 == 0:
            for k in range(0, len(qs), 2):
                c_, tg = qs[k], qs[k + 1]
                close(c_, i)
                if tg not in acc:
                    a, b = form[c_], form[tg]
                    form[tg] = (a[0] ^ b[0], a[1] ^ b[1])
        elif op == 'SWAP' and len(qs) % 2 == 0:
            for k in range(0, len(qs), 2):
                a, b = qs[k], qs[k + 1]
                close(a, i); close(b, i)
                form[a], form[b] = form[b], form[a]
        elif op == 'CCX' and len(qs) == 3:
            a, b, tg = qs
            close(a, i); close(b, i)
            if tg not in acc:
                acc[tg] = dict(start=i, entries=[])
            acc[tg]['entries'].append((i, form[a], form[b], tg))
        elif op in DIAG:
            for q in qs:
                close(q, i)
        elif op in DISCARD:
            for q in qs:
                close(q, i)
                discards.append((i, q))
                t.fresh(q, i)
        else:
            for q in qs:
                close(q, i)
                t.fresh(q, i)
    end = ops[-1][0] + 1 if ops else 0
    for q in list(acc):
        close(q, end)
    return intervals, discards, snaps, t.birth


# ── Linear algebra on snapshots ──
def echelon(vecs):
    varidx, piv = {}, {}
    for idx, (q, fs) in enumerate(vecs):
        m = 0
        for v in fs:
            m |= 1 << varidx.setdefault(v, len(varidx))
        c = 1 << idx
        while m:
            hb = m.bit_length() - 1
            if hb in piv:
                pm, pc = piv[hb]; m ^= pm; c ^= pc
            else:
                piv[hb] = (m, c); break
    return varidx, piv, vecs


def represent(ech, fs):
    varidx, piv, vecs = ech
    m = 0
    for v in fs:
        if v not in varidx:
            return None
        m |= 1 << varidx[v]
    c = 0
    while m:
        hb = m.bit_length() - 1
        if hb not in piv:
            return None
        pm, pc = piv[hb]; m ^= pm; c ^= pc
    return [vecs[k][0] for k in range(len(vecs)) if c >> k & 1]


def basis(snap, excl):
    forms, accset = snap
    return {q: f for q, f in forms.items() if q not in excl and q not in accset}


def make_ech(forms):
    return echelon([(q, f[0]) for q, f in sorted(forms.items()) if f[0]])


# ── Candidate groups ──
def nprod(P):
    return len(dickson(P)[0])


def candidates(intervals, maxrun=10, maxspan=4000, pair_window=300, maxpairs=200000):
    """Return groups: dict(targets, entries per target, plan, saving). plan: list of (poly, target tuple)."""
    out = []
    by_entry = []
    for (T, ents, s, e) in intervals:
        n = len(ents)
        for a in range(n):
            P = [set(), set(), 0]
            for b in range(a, min(n, a + maxrun)):
                if ents[b][0] - ents[a][0] > maxspan:
                    break
                add_into(P, mul(ents[b][1], ents[b][2]))
                k = b - a + 1
                if k >= 2:
                    h = nprod(P)
                    if h < k:
                        out.append(dict(targets=(T,), runs={T: ents[a:b + 1]}, iv={T: (s, e)},
                                        plan=[([set(P[0]), set(P[1]), P[2]], (T,))], saving=k - h))
        for en in ents:
            by_entry.append((en[0], T, s, e, en))
    # pairs: entries of two targets within pair_window lines sharing a variable
    by_entry.sort(key=lambda r: r[0])
    seen = set()
    pairs = 0
    lines = [r[0] for r in by_entry]
    import bisect
    ivmap = {}
    for (T, ents, s, e) in intervals:
        ivmap[(T, s)] = (ents, s, e)
    for idx, (l1, T1, s1, e1, en1) in enumerate(by_entry):
        v1 = en1[1][0] | en1[2][0]
        hi = bisect.bisect_right(lines, l1 + pair_window)
        for jdx in range(idx + 1, hi):
            l2, T2, s2, e2, en2 = by_entry[jdx]
            if T2 == T1 or (T1, s1, T2, s2) in seen or (T2, s2, T1, s1) in seen:
                continue
            if not (v1 & (en2[1][0] | en2[2][0])):
                continue
            if max(s1, s2) >= min(e1, e2):
                continue
            seen.add((T1, s1, T2, s2))
            pairs += 1
            if pairs > maxpairs:
                break
            lo, up = min(l1, l2) - pair_window, max(l1, l2) + pair_window
            r1 = [x for x in ivmap[(T1, s1)][0] if lo <= x[0] <= up][:maxrun]
            r2 = [x for x in ivmap[(T2, s2)][0] if lo <= x[0] <= up][:maxrun]
            f1, f2 = poly_of(r1), poly_of(r2)
            f12 = [f1[0] ^ f2[0], f1[1] ^ f2[1], f1[2] ^ f2[2]]
            h1, h2, h12 = nprod(f1), nprod(f2), nprod(f12)
            k = len(r1) + len(r2)
            opts = [(h1 + h2, [(f1, (T1,)), (f2, (T2,))]),
                    (h2 + h12, [(f2, (T1, T2)), (f12, (T1,))]),
                    (h1 + h12, [(f1, (T1, T2)), (f12, (T2,))])]
            best = min(opts, key=lambda o: o[0])
            if best[0] < k:
                out.append(dict(targets=(T1, T2), runs={T1: r1, T2: r2}, iv={T1: (s1, e1), T2: (s2, e2)},
                                plan=best[1], saving=k - best[0]))
        if pairs > maxpairs:
            break
    return out


# ── Validation and emission ──
def group_lines(g):
    return sorted(x[0] for r in g['runs'].values() for x in r)


def needed_lines(g, discards_sorted):
    import bisect
    ls = group_lines(g)
    lo, hi = ls[0], ls[-1]
    dl = [d[0] for d in discards_sorted]
    a, b = bisect.bisect_left(dl, lo), bisect.bisect_right(dl, hi)
    return set(ls) | {discards_sorted[k][0] for k in range(a, b)}


def emit_term(snapforms, poly, targets):
    """Return list of kmx lines that XOR poly into each target, or None if not constructible here."""
    prods, rem = dickson(poly)
    T0 = targets[0]
    out = []
    pre = [f'CX q{T0} q{T}' for T in targets[1:]]
    for (L, M) in prods:
        forms = dict(snapforms)
        best = None
        ech = make_ech(forms)
        RL = represent(ech, L[0])
        if not RL:
            return None
        for a in sorted(RL, key=lambda q: len(forms[q][0])):
            opsA = [f'CX q{q} q{a}' for q in RL if q != a]
            cL = 0
            for q in RL:
                cL ^= forms[q][1]
            if cL != L[1]:
                opsA.append(f'X q{a}')
            f2 = dict(forms); f2[a] = L
            RM = represent(make_ech(f2), M[0])
            if not RM:
                continue
            cands = [b for b in RM if b != a]
            if not cands:
                continue
            b = min(cands, key=lambda q: len(f2[q][0]))
            opsB = [f'CX q{q} q{b}' for q in RM if q != b]
            cM = 0
            for q in RM:
                cM ^= f2[q][1]
            if cM != M[1]:
                opsB.append(f'X q{b}')
            best = opsA + opsB + [f'CCX q{a} q{b} q{T0}'] + opsB[::-1] + opsA[::-1]
            break
        if best is None:
            return None
        out += best
    if rem[0] or rem[1]:
        R = represent(make_ech(snapforms), rem[0]) if rem[0] else []
        if R is None:
            return None
        out += [f'CX q{q} q{T0}' for q in R]
        c = rem[1]
        for q in R:
            c ^= snapforms[q][1]
        if c:
            out.append(f'X q{T0}')
    return pre + out + pre[::-1]


def term_factors(poly):
    """Affine forms whose span carries the polynomial's value (Dickson factors and remainder)."""
    prods, rem = dickson(poly)
    fs = [f for pm in prods for f in pm]
    if rem[0]:
        fs.append(rem)
    return fs


def validate(g, snaps, discards_sorted, birth):
    """Pick a reference time tau and placements; return dict(line -> block lines, removed lines) or None."""
    import bisect
    targets = g['targets']

    def excl_at(line):
        return {T for T in targets if g['iv'][T][0] <= line}
    ls = group_lines(g)
    dl = [d[0] for d in discards_sorted]
    entries = [x for r in g['runs'].values() for x in r]

    def born_ok(forms, line):
        return all(birth[v] < line for f in forms for v in f[0])

    def movable(forms, p, tau):
        if not born_ok(forms, min(p, tau)):
            return False
        lo, hi = min(p, tau), max(p, tau)
        for k in range(bisect.bisect_left(dl, lo), bisect.bisect_left(dl, hi)):
            h, m = discards_sorted[k]
            ech = make_ech(basis(snaps[h], excl_at(h) | {m}))
            for f in forms:
                if f[0] and represent(ech, f[0]) is None:
                    return False
        return True

    def inside(p, ts):
        return all(g['iv'][T][0] <= p < g['iv'][T][1] for T in ts)

    for tau in sorted(ls, key=lambda l: abs(l - ls[len(ls) // 2])):
        if not all(movable([e[1], e[2]], e[0], tau) for e in entries):
            continue
        blocks = defaultdict(list)
        ok = True
        for poly, ts in g['plan']:
            if not poly[0] and not poly[1] and not poly[2]:
                continue
            fac = term_factors(poly)
            placed = False
            for p in sorted(ls, key=lambda l: abs(l - tau)):
                if not inside(p, ts) or not movable(fac, p, tau):
                    continue
                code = emit_term(basis(snaps[p], excl_at(p)), poly, ts)
                if code is not None:
                    blocks[p] += code
                    placed = True
                    break
            if not placed:
                ok = False
                break
        if ok:
            return dict(blocks=dict(blocks), removed=set(ls), span=(ls[0], ls[-1]))
    return None


def apply(lines, accepted):
    removed = set().union(*[a['removed'] for a in accepted]) if accepted else set()
    blocks = defaultdict(list)
    for a in accepted:
        for p, b in a['blocks'].items():
            blocks[p] += b
    out = []
    for i, l in enumerate(lines):
        if i in blocks:
            out += blocks[i]
        if i not in removed:
            out.append(l)
    return out


def count_ccx(lines):
    return sum(1 for l in lines if l.split('#', 1)[0].strip().split(None, 1)[:1] == ['CCX'])


def optimise_round(lines, log=print, time_left=lambda: 1e9, **kw):
    ops = parse(lines)
    intervals, discards, _, birth = run(ops)
    cands = candidates(intervals, **kw)
    cands.sort(key=lambda g: (-g['saving'], group_lines(g)[-1] - group_lines(g)[0]))
    log(f'  intervals {len(intervals)}  candidate groups {len(cands)}  '
        f'potential {sum(g["saving"] for g in cands)}')
    if not cands:
        return lines, []
    # keep a non-overlapping shortlist first, so the snapshot pass stays small
    chosen, spans = [], []
    for g in cands:
        lo, hi = group_lines(g)[0], group_lines(g)[-1]
        if any(not (hi < a or lo > b) for a, b in spans):
            continue
        chosen.append(g); spans.append((lo, hi))
    discards.sort()
    want = set()
    for g in chosen:
        want |= needed_lines(g, discards)
    _, _, snaps, birth2 = run(ops, want=want)
    assert birth2 == birth
    accepted = []
    for g in chosen:
        if time_left() < 0:
            break
        v = validate(g, snaps, discards, birth)
        if v is not None:
            v['saving'] = g['saving']; v['targets'] = g['targets']
            accepted.append(v)
    log(f'  shortlisted {len(chosen)}  accepted {len(accepted)}  saving {sum(a["saving"] for a in accepted)}')
    if not accepted:
        return lines, []
    new = apply(lines, accepted)
    assert count_ccx(lines) - count_ccx(new) == sum(a['saving'] for a in accepted), 'CCX accounting'
    return new, [dict(saving=a['saving'], targets=list(a['targets']), span=list(a['span'])) for a in accepted]


def optimise(lines, rounds=8, log=print, time_left=lambda: 1e9, **kw):
    history = []
    for r in range(rounds):
        log(f'round {r}')
        lines, acc = optimise_round(lines, log=log, time_left=time_left, **kw)
        history += acc
        if not acc or time_left() < 0:
            break
    return lines, history


# ── Self-test: classical simulation of X/CX/CCX/SWAP circuits ──
def simulate(lines, n, x):
    s = list(x)
    for _, op, qs in parse(lines):
        if op == 'X':
            for q in qs: s[q] ^= 1
        elif op == 'CX':
            for k in range(0, len(qs), 2): s[qs[k + 1]] ^= s[qs[k]]
        elif op == 'CCX':
            s[qs[2]] ^= s[qs[0]] & s[qs[1]]
        elif op == 'SWAP':
            s[qs[0]], s[qs[1]] = s[qs[1]], s[qs[0]]
    return s


def self_test(seed=1, trials=300):
    import random, itertools
    rnd = random.Random(seed)
    # Dickson identity
    for _ in range(trials):
        n = rnd.randint(2, 9)
        P = [set(), set(), rnd.randint(0, 1)]
        for _ in range(rnd.randint(1, 12)):
            u, v = rnd.sample(range(n), 2); P[0] ^= {(min(u, v), max(u, v))}
        P[1] = {u for u in range(n) if rnd.random() < .4}
        prods, rem = dickson(P)
        for _ in range(16):
            x = [rnd.randint(0, 1) for _ in range(n)]
            s = rem[1] ^ (sum(x[v] for v in rem[0]) & 1)
            for L, M in prods:
                s ^= ((sum(x[v] for v in L[0]) + L[1]) & 1) & ((sum(x[v] for v in M[0]) + M[1]) & 1)
            assert s == evaluate(P, x)
    # Random wasteful circuits: equivalence on every input
    found = 0
    for tr in range(trials):
        n = rnd.randint(5, 8)
        lines = []
        for _ in range(rnd.randint(8, 30)):
            r = rnd.random()
            if r < .45:
                a, b = rnd.sample(range(n), 2); lines.append(f'CX q{a} q{b}')
            elif r < .9:
                a, b, c = rnd.sample(range(n), 3); lines.append(f'CCX q{a} q{b} q{c}')
            elif r < .95:
                lines.append(f'X q{rnd.randrange(n)}')
            else:
                a, b = rnd.sample(range(n), 2); lines.append(f'SWAP q{a} q{b}')
        new, hist = optimise(lines, rounds=4, log=lambda *a: None, maxspan=100, pair_window=30)
        found += sum(h['saving'] for h in hist)
        assert count_ccx(lines) - count_ccx(new) == sum(h['saving'] for h in hist)
        for x in itertools.product((0, 1), repeat=n):
            assert simulate(lines, n, x) == simulate(new, n, x), (lines, new)
    return found
'''
(ROOT / 'qmerge.py').write_text(QMERGE_SOURCE)
sys.path.insert(0, str(ROOT))
import importlib, qmerge; importlib.reload(qmerge)
import itertools
t0 = time.time()
print('self-test: savings found on random circuits', qmerge.self_test(1, 400) + qmerge.self_test(7, 400),
      f'({time.time() - t0:.1f} s)')
EHAN = ['CX q1 q4', 'CCX q2 q4 q5', 'CX q1 q4', 'CX q2 q4', 'CX q0 q4', 'CCX q3 q4 q5', 'CX q0 q4', 'CX q3 q4',
        'CX q1 q0', 'CCX q3 q0 q2', 'CX q1 q2', 'CX q1 q0']
new, hist = qmerge.optimise(EHAN, log=lambda *a: None)
assert qmerge.count_ccx(new) == 2 and all(qmerge.simulate(EHAN, 6, x) == qmerge.simulate(new, 6, x)
                                         for x in itertools.product((0, 1), repeat=6))
print('Ehan window 2 pattern: 3 CCX -> 2 CCX, equivalent on all 64 inputs')

## Scan

Each circuit goes `kmb → kmx text`; a round re-reads the text, finds candidate groups, keeps a non-overlapping
shortlist, validates each at concrete placements and applies the accepted ones. Rounds repeat until nothing is found.
A round-trip `kmx → kmb` must reproduce the original statistics before any rewrite counts.

In [ ]:
SCANNED = {}
def scan(task):
    src = BASE / f'{task}.kmb'
    c = km.Circuit.from_file(str(src), format='kmb')
    base_stats = c.operation_statistics()
    text = ROOT / f'{task}_base.kmx'
    c.to_file(str(text), format='kmx')
    rt = km.Circuit.from_file(str(text), format='kmx')
    assert rt.num_qubits == c.num_qubits and rt.operation_statistics().num_toffoli_like == base_stats.num_toffoli_like
    lines = text.read_text().splitlines()
    ops = qmerge.parse(lines)
    print(f'{task}: {c.num_qubits} qubits, {len(lines)} lines, ops {qmerge.stats(ops)}', flush=True)
    del c, rt
    cfg = dict(SCAN[task]); minutes = cfg.pop('minutes'); rounds = cfg.pop('rounds')
    deadline = time.time() + 60 * minutes
    t0 = time.time()
    new, hist = qmerge.optimise(lines, rounds=rounds, time_left=lambda: deadline - time.time(),
                                log=lambda s: print(f'  [{(time.time() - t0) / 60:5.1f} min] {s}', flush=True), **cfg)
    saved = sum(h['saving'] for h in hist)
    print(f'{task}: {len(hist)} rewrites, {saved} CCX saved, {(time.time() - t0) / 60:.1f} min', flush=True)
    SCANNED[task] = dict(rewrites=hist, saved=saved)
    if saved:
        out = ROOT / f'{task}_merged.kmx'
        out.write_text('\n'.join(new) + '\n')
        m = km.Circuit.from_file(str(out), format='kmx')
        assert m.num_qubits <= km.Circuit.from_file(str(src), format='kmb').num_qubits
        m.to_file(str(ROOT / f'{task}_merged.kmb'), format='kmb')
        SCANNED[task]['candidate'] = str(ROOT / f'{task}_merged.kmb')
    (ROOT / 'merge_report.json').write_text(json.dumps(SCANNED, indent=1))

scan('invsqrt')

In [ ]:
scan('dqi')

## Official verification

Only a member the scanner changed is re-verified (invsqrt ≈ 13 min over the whole domain, DQI ≈ 3 min); unchanged
members keep notebook 07's verified results, matched by sha256. A candidate is used only if it passes, keeps the
qubit count and lowers the cost.

In [ ]:
FINAL = ROOT / 'final'; shutil.rmtree(FINAL, ignore_errors=True); shutil.copytree(BASE, FINAL)
RESULTS = {t: BASE_RESULTS[t] for t in ('invsqrt', 'dqi', 'dfthcsos')}
CHOICE = {}
for task, verifier, spec in (('invsqrt', vi, lambda: vi.ProblemSpecInvSqrt.for_n_p(10)),
                             ('dqi', vd, lambda: vd.ProblemSpecDQI.benchmark_instance())):
    cand = SCANNED.get(task, {}).get('candidate')
    if not cand:
        CHOICE[task] = 'base (no rewrite found)'; continue
    t0 = time.time()
    r = verifier.verify(km.Circuit.from_file(cand, format='kmb'), spec(), salt=SALT, max_workers=WORKERS)
    base = BASE_RESULTS[task]
    print(f"{task}: {r['status']}  cost {float(r['score']):.8g} (base {float(base['score']):.8g})  "
          f"T {r['toffoli_count']} (base {base['toffoli_count']})  Q {r['num_qubits']}  {(time.time() - t0) / 60:.1f} min",
          flush=True)
    good = (r['status'] == 'PASS' and r['num_qubits'] <= base['num_qubits'] and float(r['score']) < float(base['score']))
    SCANNED[task]['verify'] = r
    if good:
        shutil.copyfile(cand, FINAL / f'{task}.kmb'); RESULTS[task] = r; CHOICE[task] = 'merged'
    else:
        CHOICE[task] = 'base (candidate rejected)'
print(CHOICE)

In [ ]:
# Chemistry is byte-identical to notebook 07; re-run its fast verifier as a guard (≈ 30 s).
chem = vc.verify(str(FINAL / 'dfthcsos'), salt=SALT)
assert chem['status'] == 'PASS' and chem['toffoli_count'] == BASE_RESULTS['dfthcsos']['toffoli_count']
RESULTS['dfthcsos'] = chem
assert {str(p): sha256(p) for p in VERIFIER_FILES} == VERIFIER_SHA, 'verifier files changed'

files = sorted(p.relative_to(FINAL).as_posix() for p in FINAL.rglob('*') if p.is_file())
assert files == MEMBERS
partial = ROOT / 'submission.zip.partial'
with zipfile.ZipFile(partial, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for name in files:
        z.write(FINAL / name, name)
with zipfile.ZipFile(partial) as z:
    assert sorted(z.namelist()) == files
os.replace(partial, ROOT / 'submission.zip')

ratios = {t: REFERENCE[t] / float(RESULTS[t]['score']) for t in RESULTS}
GM = math.prod(ratios.values()) ** (1 / 3)
GM0 = math.prod(REFERENCE[t] / float(BASE_RESULTS[t]['score']) for t in RESULTS) ** (1 / 3)
summary = dict(choice=CHOICE, ratios=ratios, expected_GM=GM, base_GM=GM0, scan=SCANNED,
               sha256={n: sha256(FINAL / n) for n in files}, submission_zip_sha256=sha256(ROOT / 'submission.zip'),
               salt=SALT, minutes=round((time.time() - NB_T0) / 60, 1))
(ROOT / 'merge_report.json').write_text(json.dumps(summary, indent=1, default=str))
for f in ('invsqrt_base.kmx', 'dqi_base.kmx', 'invsqrt_merged.kmx', 'dqi_merged.kmx'):
    (ROOT / f).unlink(missing_ok=True)
shutil.rmtree(BASE); shutil.rmtree(FINAL)
print(json.dumps({t: round(v, 6) for t, v in ratios.items()}))
print(f'QUADRATIC MERGE DONE | GM {GM:.6f} (base {GM0:.6f}, {GM - GM0:+.6f}) | {summary["minutes"]} min')

# Sources and credits

- **Ehan Mallick**: the idea this notebook automates (XOR-only windows as GF(2) quadratic forms, Dickson rank/2,
  products shared by CX fan-out) and the base invsqrt circuit qf2
  ([invsqrt 20.65: two 3-to-2 AND merges on y6225](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225), Apache 2.0).
- **yoyobar**: invsqrt y6225, which qf2 patches, and the DQI circuit in the portfolio
  ([LB 22.99 invsqrt 101 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits)); y6225 builds on
  **Ahmed Berat Özer**, **busyaprime (Akmal Xodarev)** and **mrmeows**.
- **SomeJoint**: the DQI decoder construction ([confluent DQI](https://www.kaggle.com/code/somejoint/quantum-circuits-20-96-local-gm-confluent-dqi));
  **busyaprime**: exact DQI rewrites ([LB 23.91003](https://www.kaggle.com/code/busyaprime/lb-23-91003-exact-dqi-rewrites-at-892-qubits)).
- **heliosli**: the public portfolio release `v59_c40h14892y5` that notebook 07 starts from
  ([qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)); its chemistry is the work of
  **Lera (marculera)**, **manwithacat** (HW-12…14 loaders), **matousakura** (GF(2) relations and the 313-Toffoli
  loaders k0..3 used here) and others credited in notebooks 01 and 07.
- **L. E. Dickson** (1901): canonical form of quadratic forms over GF(2).
- Ours: the scanner (symbolic affine tracking, span-checked movement past `HMR`/`R`, pair sharing), its tests and the
  packaging.